Data source: https://www.wsj.com/graphics/djia-components-history/

In [2]:
import json
import re
import requests

import numpy as np
import pandas as pd

In [3]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'
}

resp = requests.get('https://g.wsj.net/djia-components-history/js/app-eda34bae.js', headers=headers)

In [4]:
resp.raise_for_status()

In [5]:
# extract data
data = re.findall(r'\[\{.+?\}\]', resp.text)[2]

# add quotes for JSON parsing
data = re.sub(r'([{,])(\w+?):', r'\1"\2":', data)

# parse
data = json.loads(data)

In [7]:
# filter keys

_map = {'yearCounts':'dates', 'company':'company'}

data = [{_map.get(key): rec[key] for key in (rec.keys() and {'company', 'yearCounts'})} for rec in data]

In [9]:
# save file
with open('dow_components.json', 'w') as outf:
    json.dump(data, outf, indent=4, sort_keys=True)

In [10]:
# reshape
allrecs = []
for rec in data:
    for date in rec['dates']:
        allrecs.append((rec['company'], date))

# create dataframe
dow = pd.DataFrame(allrecs)
dow.columns = ['company', 'date']
dow['date'] = pd.to_datetime(dow['date'])
dow = dow.sort_values(['date', 'company'])

In [12]:
dow

,company,date
0,American Cotton Oil,1896-05-26
9,American Sugar,1896-05-26
51,American Tobacco,1896-05-26
72,Chicago Gas,1896-05-26
7,Distilling & Cattle Feeding,1896-05-26
...,...,...
1479,UnitedHealth Group,2015-03-06
1450,Verizon,2015-03-06
1483,Visa,2015-03-06
1359,Wal-Mart,2015-03-06


In [14]:
smpl = dow[dow['date']>='1925-12-31']

In [15]:
smpl.groupby('date')['company'].count()

date
1925-12-31    20
1927-03-16    20
1928-10-01    30
1929-01-08    29
1929-09-14    30
1930-01-29    30
1930-07-18    30
1932-05-26    30
1933-08-15    30
1934-08-13    30
1935-11-20    30
1937-01-08    30
1939-03-14    30
1956-07-03    30
1959-04-22    30
1959-06-01    30
1976-04-21    30
1976-08-09    30
1979-06-29    30
1982-08-30    30
1985-09-19    30
1985-10-30    30
1986-07-08    30
1987-03-12    30
1988-12-16    30
1991-05-06    30
1994-04-20    30
1997-03-17    30
1999-01-04    30
1999-11-01    30
2002-04-08    30
2003-01-27    30
2004-04-08    30
2005-11-21    30
2008-02-19    30
2008-09-22    30
2009-06-08    30
2012-09-24    30
2013-09-23    30
2015-03-06    30
Name: company, dtype: int64

### Add recent data from Wikipedia

In [3]:
from bs4 import BeautifulSoup

In [4]:
headers = {'User-Agent': 'Educational web scraper - contact: nstoffma@iu.edu'}

In [5]:
resp = requests.get('https://en.wikipedia.org/wiki/Historical_components_of_the_Dow_Jones_Industrial_Average',
                    headers=headers)

In [6]:
tbls = re.findall(r'<h2 id=".+?">.+?</table>', resp.text, re.DOTALL)

In [7]:
len(tbls)

61

In [8]:
dow = []
for tbl in tbls:
    soup = BeautifulSoup(tbl)
    comps = pd.DataFrame([td.text.strip() for td in soup.find_all('td')],
                         columns=['company'])
    comps['date'] = soup.find('h2').text.strip()
    dow.append(comps)

In [9]:
dow = pd.concat(dow).reset_index(drop=True)

dow['date'] = pd.to_datetime(dow['date'], errors='coerce')

dow = dow.dropna()

In [10]:
dow

,company,date
0,3M Company,2024-11-08
1,"The Goldman Sachs Group, Inc.",2024-11-08
2,Nvidia Corporation ↑,2024-11-08
3,"Amazon.com, Inc.",2024-11-08
4,"The Home Depot, Inc.",2024-11-08
...,...,...
1673,The Laclede Gas Company ↑,1896-05-26
1674,The United States Leather Company (Preferred) ↑,1896-05-26
1675,Chicago Gas Light and Coke Company ↑,1896-05-26
1676,National Lead Company ↑,1896-05-26


In [11]:
# drop companies that were dropped on a given date
dow = dow[~dow['company'].str.endswith('↓')]

# clean up names
dow = dow[~(dow['company'] == 'Dropped from Average')]
dow['company'] = dow['company'].str.replace(r' ?\(.+?\)', '', regex=True)
dow['company'] = dow['company'].str.replace(' ↑', '')
dow['company'] = dow['company'].str.replace(' †', '')
dow['company'] = dow['company'].str.replace(r' ?\[\d\]', '', regex=True)

# drop empty company names
dow = dow[dow['company'].str.len() > 0]

dow = dow.sort_values(['date', 'company'])

dow = dow.set_index('date').squeeze()

In [12]:
dow

date
1896-05-26              American Tobacco Company
1896-05-26    Chicago Gas Light and Coke Company
1896-05-26       Distilling & Cattle Feeding Co.
1896-05-26              General Electric Company
1896-05-26                 National Lead Company
                             ...                
2024-11-08               The Walt Disney Company
2024-11-08       UnitedHealth Group Incorporated
2024-11-08           Verizon Communications Inc.
2024-11-08                             Visa Inc.
2024-11-08                          Walmart Inc.
Name: company, Length: 1429, dtype: object

In [13]:
# one row per firm: first and last date in the Dow history
if isinstance(dow, pd.Series):
    dow_dates = dow.rename('company').reset_index()
else:
    dow_dates = dow.copy()
    if 'date' not in dow_dates.columns:
        dow_dates = dow_dates.reset_index()

firm_date_ranges = (
    dow_dates[['date', 'company']]
    .dropna()
    .drop_duplicates()
    .groupby('company', as_index=False)
    .agg(start_date=('date', 'min'), end_date=('date', 'max'))
    .sort_values(['start_date', 'company'])
    .reset_index(drop=True)
)

# optional: mark firms that are still in the index at the latest observed date
latest_date = dow_dates['date'].max()
firm_date_ranges['is_current_member'] = firm_date_ranges['end_date'].eq(latest_date)

firm_date_ranges



,company,start_date,end_date,is_current_member
0,American Tobacco Company,1896-05-26,1982-08-30,False
1,Chicago Gas Light and Coke Company,1896-05-26,1896-12-23,False
2,Distilling & Cattle Feeding Co.,1896-05-26,1896-05-26,False
3,General Electric Company,1896-05-26,2017-09-01,False
4,National Lead Company,1896-05-26,1915-07-29,False
...,...,...,...,...
151,Amgen Inc.,2020-08-31,2024-11-08,True
152,"Salesforce, Inc.",2020-08-31,2024-11-08,True
153,"Amazon.com, Inc.",2024-02-26,2024-11-08,True
154,Nvidia Corporation,2024-11-08,2024-11-08,True


In [14]:
firm_date_ranges[firm_date_ranges['is_current_member']]

,company,start_date,end_date,is_current_member
82,International Business Machines Corporation,1932-05-26,2024-11-08,True
85,The Coca-Cola Company,1932-05-26,2024-11-08,True
86,The Procter & Gamble Company,1932-05-26,2024-11-08,True
100,"Merck & Co., Inc.",1979-06-29,2024-11-08,True
101,American Express Company,1982-08-30,2024-11-08,True
103,Chevron Corporation,1985-10-30,2024-11-08,True
104,McDonald's Corporation,1985-10-30,2024-11-08,True
108,The Boeing Company,1987-03-12,2024-11-08,True
111,Caterpillar Inc.,1991-05-06,2024-11-08,True
113,The Walt Disney Company,1991-05-06,2024-11-08,True


In [15]:
# current Dow members from the date-range table
if 'firm_date_ranges' not in globals():
    if isinstance(dow, pd.Series):
        dow_dates = dow.rename('company').reset_index()
    else:
        dow_dates = dow.copy()
        if 'date' not in dow_dates.columns:
            dow_dates = dow_dates.reset_index()

    firm_date_ranges = (
        dow_dates[['date', 'company']]
        .dropna()
        .drop_duplicates()
        .groupby('company', as_index=False)
        .agg(start_date=('date', 'min'), end_date=('date', 'max'))
    )
    latest_date = dow_dates['date'].max()
    firm_date_ranges['is_current_member'] = firm_date_ranges['end_date'].eq(latest_date)

current_dow = (
    firm_date_ranges.loc[firm_date_ranges['is_current_member'], ['company', 'start_date', 'end_date']]
    .sort_values('company')
    .reset_index(drop=True)
)

current_dow


,company,start_date,end_date
0,3M Company,2003-01-27,2024-11-08
1,"Amazon.com, Inc.",2024-02-26,2024-11-08
2,American Express Company,1982-08-30,2024-11-08
3,Amgen Inc.,2020-08-31,2024-11-08
4,Apple Inc.,2015-03-19,2024-11-08
5,Caterpillar Inc.,1991-05-06,2024-11-08
6,Chevron Corporation,1985-10-30,2024-11-08
7,"Cisco Systems, Inc.",2009-06-08,2024-11-08
8,Honeywell International Inc.,2005-11-21,2024-11-08
9,International Business Machines Corporation,1932-05-26,2024-11-08


In [16]:
# pull CRSP names with an as-of date no later than Dec 31 of previous year
import wrds
import pandas as pd

if 'db' not in globals():
    db = wrds.Connection()

prev_year_end = pd.Timestamp.today().normalize().replace(month=12, day=31, year=pd.Timestamp.today().year - 1)

# Try msenames first
try:
    max_df = db.raw_sql("""
        select max(nameendt) as max_name_date
        from crsp.msenames
        where shrcd in (10, 11)
    """)
    max_name_date = pd.to_datetime(max_df.loc[0, 'max_name_date'])
    asof_date = min(prev_year_end, max_name_date)

    crsp_names = db.raw_sql(f"""
        select permno, comnam, ticker, namedt, nameendt, shrcd
        from crsp.msenames
        where shrcd in (10, 11)
          and namedt <= '{asof_date.date()}'
          and nameendt >= '{asof_date.date()}'
    """)
except Exception:
    max_df = db.raw_sql("""
        select max(nameenddt) as max_name_date
        from crsp.stocknames
        where shrcd in (10, 11)
    """)
    max_name_date = pd.to_datetime(max_df.loc[0, 'max_name_date'])
    asof_date = min(prev_year_end, max_name_date)

    crsp_names = db.raw_sql(f"""
        with ranked as (
            select
                permno,
                comnam,
                ticker,
                namedt,
                nameenddt,
                shrcd,
                row_number() over (
                    partition by permno
                    order by
                        case when namedt <= '{asof_date.date()}' and nameenddt >= '{asof_date.date()}' then 0 else 1 end,
                        nameenddt desc,
                        namedt desc
                ) as rn
            from crsp.stocknames
            where shrcd in (10, 11)
        )
        select permno, comnam, ticker, namedt, nameenddt as nameendt, shrcd
        from ranked
        where rn = 1
    """)

crsp_names = (
    crsp_names
    .dropna(subset=['permno', 'comnam'])
    .drop_duplicates(['permno', 'comnam'])
    .reset_index(drop=True)
)

print(f'prev_year_end cap: {prev_year_end.date()}')
print(f'CRSP max name date: {max_name_date.date()}')
print(f'asof_date used: {asof_date.date()}')
print(f'Rows pulled: {len(crsp_names):,}')
crsp_names.head()


Loading library list...
Done
prev_year_end cap: 2025-12-31
CRSP max name date: 2024-12-31
asof_date used: 2024-12-31
Rows pulled: 3,806


,permno,comnam,ticker,namedt,nameendt,shrcd
0,10026,J & J SNACK FOODS CORP,JJSF,2024-06-20,2024-12-31,11
1,10028,ENVELA CORP,ELA,2020-02-24,2024-12-31,11
2,10032,PLEXUS CORP,PLXS,2019-09-12,2024-12-31,11
3,10044,ROCKY MOUNTAIN CHOC FAC INC NEW,RMCF,2021-02-23,2024-12-31,11
4,10066,FRANKLIN WIRELESS CORP,FKWL,2024-06-17,2024-12-31,11


In [17]:
# fuzzy match current Dow company names to CRSP names
import re

try:
    from rapidfuzz import fuzz, process
    USE_RAPIDFUZZ = True
except Exception:
    from difflib import SequenceMatcher
    USE_RAPIDFUZZ = False

def normalize_name(x):
    x = str(x).upper()
    x = re.sub(r'&', ' AND ', x)
    x = re.sub(r'[^A-Z0-9 ]+', ' ', x)
    x = re.sub(r'\b(THE|INCORPORATED|INC|CORPORATION|CORP|COMPANY|CO|HOLDINGS|HOLDING|GROUP|PLC|LLC|LTD|LIMITED|SA|NV)\b', ' ', x)
    x = re.sub(r'\s+', ' ', x).strip()
    return x

crsp_match_base = crsp_names[['permno', 'comnam', 'ticker']].copy()
crsp_match_base['comnam_norm'] = crsp_match_base['comnam'].map(normalize_name)

choices_raw = crsp_match_base['comnam'].tolist()
choices_norm = crsp_match_base['comnam_norm'].tolist()

rows = []
for company in current_dow['company']:
    company_norm = normalize_name(company)

    if USE_RAPIDFUZZ:
        m_raw = process.extractOne(company, choices_raw, scorer=fuzz.WRatio)
        m_norm = process.extractOne(company_norm, choices_norm, scorer=fuzz.WRatio)

        best = m_raw
        match_on = 'raw'
        if m_norm and (not m_raw or m_norm[1] > m_raw[1]):
            best = m_norm
            match_on = 'normalized'

        if best is None:
            rows.append({'company': company})
            continue

        match_value, score, idx = best
    else:
        def score_fn(a, b):
            return int(100 * SequenceMatcher(None, a, b).ratio())

        raw_scores = [score_fn(company, c) for c in choices_raw]
        norm_scores = [score_fn(company_norm, c) for c in choices_norm]

        raw_idx = max(range(len(raw_scores)), key=raw_scores.__getitem__)
        norm_idx = max(range(len(norm_scores)), key=norm_scores.__getitem__)

        if norm_scores[norm_idx] > raw_scores[raw_idx]:
            idx = norm_idx
            score = norm_scores[norm_idx]
            match_value = choices_norm[idx]
            match_on = 'normalized'
        else:
            idx = raw_idx
            score = raw_scores[raw_idx]
            match_value = choices_raw[idx]
            match_on = 'raw'

    hit = crsp_match_base.iloc[idx]
    rows.append({
        'company': company,
        'matched_comnam': hit['comnam'],
        'permno': int(hit['permno']),
        'ticker': hit['ticker'],
        'score': int(score),
        'match_on': match_on
    })

dow_permno_matches = pd.DataFrame(rows).sort_values(['score', 'company'], ascending=[False, True]).reset_index(drop=True)

# quick review tables
dow_permno_matches


,company,matched_comnam,permno,ticker,score,match_on
0,3M Company,3M CO,22592,MMM,100,normalized
1,"Amazon.com, Inc.",AMAZON COM INC,84788,AMZN,100,normalized
2,American Express Company,AMERICAN EXPRESS CO,59176,AXP,100,normalized
3,Amgen Inc.,AMGEN INC,14008,AMGN,100,normalized
4,Apple Inc.,APPLE INC,14593,AAPL,100,normalized
5,Caterpillar Inc.,CATERPILLAR INC,18542,CAT,100,normalized
6,"Cisco Systems, Inc.",CISCO SYSTEMS INC,76076,CSCO,100,normalized
7,Honeywell International Inc.,HONEYWELL INTERNATIONAL INC,10145,HON,100,normalized
8,JPMorgan Chase & Co.,JPMORGAN CHASE & CO,47896,JPM,100,normalized
9,Johnson & Johnson,JOHNSON & JOHNSON,22111,JNJ,100,normalized


In [24]:
fdir = '/Users/nstoffma/Documents/GitHub/qf_data'

dow_permno_matches.reindex(['company', 'ticker', 'permno'], axis=1).to_csv(f'{fdir}/dow_current_components.csv', index=False)